In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("sql-analysis-final").getOrCreate()

GOLD_PATH = "/home/jovyan/work/submission/output/gold"

fact_order_items = spark.read.parquet(f"{GOLD_PATH}/fact_order_items")
dim_products = spark.read.parquet(f"{GOLD_PATH}/dim_products")
dim_sellers = spark.read.parquet(f"{GOLD_PATH}/dim_sellers")

In [9]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_products.createOrReplaceTempView("dim_products")
dim_sellers.createOrReplaceTempView("dim_sellers")

In [10]:
spark.sql("""
-- Revenue Trend Analysis with Ranking

WITH monthly_data AS (
    SELECT
        p.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        COUNT(*) AS transactions,
        SUM(f.price + f.freight_value) AS monthly_revenue
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY p.product_category_name, year, month
),

filtered_data AS (
    SELECT *
    FROM monthly_data
    WHERE transactions >= 10
),

ranked_data AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM filtered_data
),

top_categories AS (
    SELECT product_category_name
    FROM filtered_data
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),

final_calc AS (
    SELECT
        r.product_category_name,
        r.year,
        r.month,
        r.monthly_revenue,
        r.monthly_rank,

        -- Month-over-month growth
        (r.monthly_revenue - LAG(r.monthly_revenue)
            OVER (PARTITION BY r.product_category_name ORDER BY r.year, r.month))
        /
        LAG(r.monthly_revenue)
            OVER (PARTITION BY r.product_category_name ORDER BY r.year, r.month)
        AS mom_growth_pct,

        -- Rolling 3-month avg
        AVG(r.monthly_revenue)
            OVER (
                PARTITION BY r.product_category_name
                ORDER BY r.year, r.month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS rolling_3m_avg_revenue

    FROM ranked_data r
    WHERE r.product_category_name IN (SELECT product_category_name FROM top_categories)
)

SELECT *
FROM final_calc
ORDER BY product_category_name, year, month
""").show()

+---------------------+----+-----+------------------+------------+--------------------+----------------------+
|product_category_name|year|month|   monthly_revenue|monthly_rank|      mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+------------------+------------+--------------------+----------------------+
|         beleza_saude|2016|   10|            5635.8|           3|                NULL|                5635.8|
|         beleza_saude|2017|    1|          17275.91|           2|  2.0653873451861315|             11455.855|
|         beleza_saude|2017|    2|30139.059999999998|           1|  0.7445714871170316|              17683.59|
|         beleza_saude|2017|    3|30776.919999999995|           5|0.021163898276853922|    26063.963333333333|
|         beleza_saude|2017|    4|          27039.82|           6|-0.12142540579109266|    29318.599999999995|
|         beleza_saude|2017|    5|          53662.87|           1|  0.9845868056813989|    37159.869999999995|
|

In [11]:
spark.sql("""
-- Seller Performance Scorecard

WITH seller_metrics AS (
    SELECT
        f.seller_key,
        s.seller_state,
        COUNT(*) AS total_orders,
        SUM(f.payment_value) AS total_revenue,

        AVG(CASE WHEN f.is_late_delivery THEN 1 ELSE 0 END) AS late_delivery_rate,
        AVG(f.days_delivery_vs_estimate) AS avg_days_vs_estimate

    FROM fact_order_items f
    JOIN dim_sellers s ON f.seller_key = s.seller_key
    GROUP BY f.seller_key, s.seller_state
),

filtered AS (
    SELECT *
    FROM seller_metrics
    WHERE total_orders >= 20
),

scored AS (
    SELECT *,
        1 - PERCENT_RANK() OVER (ORDER BY late_delivery_rate) AS on_time_pctl,
        1 - PERCENT_RANK() OVER (ORDER BY avg_days_vs_estimate) AS speed_pctl,
        PERCENT_RANK() OVER (ORDER BY total_revenue) AS revenue_pctl
    FROM filtered
),

final_score AS (
    SELECT *,
        (on_time_pctl * 0.4 +
         speed_pctl * 0.3 +
         revenue_pctl * 0.3) AS composite_score
    FROM scored
)

SELECT *,
    RANK() OVER (ORDER BY composite_score DESC) AS overall_rank
FROM final_score
ORDER BY overall_rank
""").show()

+--------------------+------------+------------+------------------+--------------------+--------------------+------------------+------------------+------------------+------------------+------------+
|          seller_key|seller_state|total_orders|     total_revenue|  late_delivery_rate|avg_days_vs_estimate|      on_time_pctl|        speed_pctl|      revenue_pctl|   composite_score|overall_rank|
+--------------------+------------+------------+------------------+--------------------+--------------------+------------------+------------------+------------------+------------------+------------+
|1968512e989c4d415...|          RJ|          58|          54027.64|                 0.0| -18.948275862068964|               1.0|0.9692982456140351|0.9331140350877193|0.9707236842105265|           1|
|c4ac09d91c477e30b...|          RS|         100|25884.280000000002|                 0.0| -17.195876288659793|               1.0|0.9298245614035088|0.8421052631578947|0.9315789473684211|           2|
|dcb9